In [ ]:
!pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

COINS = {"BTC-USD": "Bitcoin", "ETH-USD": "Ethereum", "BNB-USD": "BNB",
         "XRP-USD": "XRP", "SOL-USD": "Solana", "ADA-USD": "Cardano", "DOGE-USD": "Dogecoin"}

raw = yf.download(list(COINS), start="2021-01-01", auto_adjust=True, progress=False)
close = raw["Close"].rename(columns=COINS).dropna()
volume = raw["Volume"].rename(columns=COINS).loc[close.index]
close.index.name = "date"
volume.index.name = "date"

# 1. Prices in long format (one row per coin per day)
close_long = close.reset_index().melt(id_vars="date", var_name="coin", value_name="close")
vol_long = volume.reset_index().melt(id_vars="date", var_name="coin", value_name="volume")
prices = close_long.merge(vol_long, on=["date", "coin"])
prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")
prices.to_csv("prices_long.csv", index=False)

rets = close.pct_change().dropna()

# 2. 30-day rolling volatility (annualised)
vol30 = (rets.rolling(30).std() * np.sqrt(365)).dropna()
rolling_vol = vol30.reset_index().melt(id_vars="date", var_name="coin", value_name="vol_30d")
rolling_vol["date"] = rolling_vol["date"].dt.strftime("%Y-%m-%d")
rolling_vol.to_csv("rolling_vol.csv", index=False)

# 3. Correlation between every pair of coins
corr = rets.corr().rename_axis("coin_a").reset_index()
corr.melt(id_vars="coin_a", var_name="coin_b", value_name="correlation").to_csv("correlation.csv", index=False)

# 4. 90-day rolling correlation with Bitcoin
btc = rets.drop(columns="Bitcoin").rolling(90).corr(rets["Bitcoin"]).dropna()
btc = btc.reset_index().melt(id_vars="date", var_name="coin", value_name="corr_90d_btc")
btc["date"] = btc["date"].dt.strftime("%Y-%m-%d")
btc.to_csv("rolling_corr_btc.csv", index=False)

# 5. Coin list for Power BI slicers
pd.DataFrame({"coin": list(COINS.values())}).to_csv("coins.csv", index=False)
print(prices.shape)

(14840, 4)
